# Fase 3.6 — teste isolado de robustez a redimensionamento

Reutiliza os dados e partições congelados da Fase 3.5, o mesmo checkpoint inicial da Fase 3.3 e os mesmos hiperparâmetros. Só a probabilidade de `RandomDownUp` no treino muda de 0,15 para 0,50. O notebook não inicia treino ao abrir, não usa o teste OpenFake e não consulta avaliações externas. Preserve todos os experimentos anteriores.


In [1]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_5', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 7.7 GB
/content livre: 204.55 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'main'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase36-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('Notebooks/fase_3/fase3_5/fase3_5_dados.py', 'Notebooks/fase_3/fase3_6/fase3_6_escala.py', 'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc35_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.5 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: dfb808ee46b962c80dd414f8a5e52de551d5abba


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'Notebooks/fase_3/fase3_5/fase3_5_dados.py'
TRAIN = CODE_ROOT / 'Notebooks/fase_3/fase3_6/fase3_6_escala.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/dfb808ee46b962c80dd414f8a5e52de551d5abba


## Execução manual

A preparação da Fase 3.5 já foi concluída. Em cada sessão, a célula 6 copia e confere os dados internos no disco temporário. A célula 7 faz o smoke; a célula 8 inicia ou retoma o treino somente quando executada. A célula 9 lê os artefatos.


In [6]:
# 6. CÓPIA LOCAL: usa o conjunto interno já congelado da Fase 3.5.
run_live('cópia local 3.6', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])


cópia local 3.6: iniciando
Cópia/verificação: 100/3000
Cópia/verificação: 200/3000
Cópia/verificação: 300/3000
Cópia/verificação: 400/3000
Cópia/verificação: 500/3000
Cópia/verificação: 600/3000
Cópia/verificação: 700/3000
Cópia/verificação: 800/3000
Cópia/verificação: 900/3000
Cópia/verificação: 1000/3000
Cópia/verificação: 1100/3000
Cópia/verificação: 1200/3000
Cópia/verificação: 1300/3000
Cópia/verificação: 1400/3000
Cópia/verificação: 1500/3000
Cópia/verificação: 1600/3000
Cópia/verificação: 1700/3000
Cópia/verificação: 1800/3000
Cópia/verificação: 1900/3000
Cópia/verificação: 2000/3000
Cópia/verificação: 2100/3000
Cópia/verificação: 2200/3000
Cópia/verificação: 2300/3000
Cópia/verificação: 2400/3000
Cópia/verificação: 2500/3000
Cópia/verificação: 2600/3000
Cópia/verificação: 2700/3000
Cópia/verificação: 2800/3000
Cópia/verificação: 2900/3000
Cópia/verificação: 3000/3000
Dados internos locais prontos: /content/fase3_5_novos_v1
cópia local 3.6: concluído em 0.1 min


In [7]:
# 7. SMOKE MANUAL: verifica dados, divisões, checkpoint e forward; não treina.
run_live('smoke 3.6', [sys.executable, '-u', str(TRAIN), 'smoke'])


smoke 3.6: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.6: 3594 719 719 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/3594
Hashes treino: 1000/3594
Hashes treino: 1500/3594
Hashes treino: 2000/3594
Hashes treino: 2500/3594
Hashes treino: 3000/3594
Hashes treino: 3500/3594
Hashes seleção: 500/719
Hashes calibração: 500/719
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Partições idênticas às da Fase 3.5 confirmadas.
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
Smoke OK: /content/drive/MyDrive/TCC_Final/modelos/fase3_6_escala_p05_v1_smoke/smoke.json
smoke 3.6: concluído em 0.2 min


In [8]:
# 8. TREINO MANUAL: inicia ou retoma somente quando você executar esta célula.
run_live('ajuste fino 3.6', [sys.executable, '-u', str(TRAIN), 'auto'])


ajuste fino 3.6: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.6: 3594 719 719 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/3594
Hashes treino: 1000/3594
Hashes treino: 1500/3594
Hashes treino: 2000/3594
Hashes treino: 2500/3594
Hashes treino: 3000/3594
Hashes treino: 3500/3594
Hashes seleção: 500/719
Hashes calibração: 500/719
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Partições idênticas às da Fase 3.5 confirmadas.
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
ajuste fino 3.6: ativo há 0.7 min; sem nova saída
[E01/12] lr=3.0e-05 | train_loss=0.4929 | val_loss=0.4186 | AUC=0.8958 | bal_acc=0.8199 | 37s  -> melhor
[E02/12] lr=3.0e-05 | train_loss=0.4824 | val_loss=0.3831 | AUC=0.9184 | bal_acc=0.8249 | 20s  -> melhor
[E03/12]

In [9]:
# 9. CONFERÊNCIA SOMENTE-LEITURA dos artefatos da variante 3.6.
import json
run_dir = ROOT / 'modelos/fase3_6_escala_p05_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt', 'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    path = run_dir / name
    print(name, 'presente' if path.is_file() else 'AUSENTE')
metrics_path = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.is_file() else 'Treino interno ainda não concluído.')


experiment_config.json presente
split_manifest.json presente
history.csv presente
last.pt presente
best.pt presente
phase3_final.pt presente
internal_metrics.json presente
training_complete.json presente
{
  "best_epoch": 12,
  "selection_auc": 0.9472354549004295,
  "calibration_bal_acc_apparent": 0.9075868801249511,
  "temperature": 0.466225802898407,
  "threshold": 0.49499999999999994,
  "train_count": 3594,
  "selection_count": 719,
  "calibration_count": 719,
  "external_used": false,
  "openfake_test_used": false
}
